# TEMA: Distribuciones condicionadas

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from empiricaldist import Pmf, Cdf


########################## CARGA DESDE ARCHIVO LOCAL ##########################
# Cargar el archivo local
data = pd.read_csv("cln_vista_agrosavia_Municipio_Funza_20260111_cleaned.csv", sep=",", encoding="utf-8")

# Ver las primeras filas
display(data.head())

,Departamento,Municipio,Cultivo,Estado,Topografia,Drenaje,Riego,Fertilizantes aplicados,Secuencial
0,CUNDINAMARCA,FUNZA,Guisantes,Por Establecer,Plano,Buen drenaje,Goteo,urea,166.0
1,CUNDINAMARCA,FUNZA,Lechuga,Por Establecer,Plano,Buen drenaje,Aspersión,15-15-15,248.0
2,CUNDINAMARCA,FUNZA,Zanahoria,Por Establecer,Plano,Buen drenaje,Aspersión,15-15-15,249.0
3,CUNDINAMARCA,FUNZA,Maíz,Por Establecer,Plano,Buen drenaje,Aspersión,15-15-15,306.0
4,CUNDINAMARCA,FUNZA,Lechuga,Establecido,Plano,Buen drenaje,Aspersión,"compostaje+quimico edafico+(dap,kcl+fertilizac...",388.0


Dado que el cultivo es X, ¿cómo se distribuye Y?
Cómo cambia una variable cuando fijo otra
no especificaron una relación real cultivo–condiciones
Pero las distribuciones condicionadas NO afirman causalidad, solo describen patrones observados.
P(Riego | Cultivo)
P(Drenaje | Cultivo)

In [3]:
# tabla de contingencia P(Riego | Cultivo)
# Se toman los 12 cultivos mas frecuentes segun la grafica de PMF de Cultivo
# Se ORDENA tabla por cultivos por frecuencia total (descendente)
# Se elimina columnas no informativas: 'No Tiene' y 'No Indica'

# Generar tabla de contingencia no normalizada
tabla_abs = (  # (frecuencias absolutas)
    data 
    .groupby("Cultivo")["Riego"]
    .value_counts()
    .unstack(fill_value=0)
)
# Ordenar
orden_cultivos = tabla_abs.sum(axis=1).sort_values(ascending=False).index
tabla_abs = tabla_abs.loc[orden_cultivos]
# Eliminar columnas no informativas
columnas_a_eliminar = ["No Tiene", "No Indica", "No indica"]
tabla_abs_limpia = tabla_abs.drop(
    columns=[c for c in columnas_a_eliminar if c in tabla_abs.columns]
)
# Dejar solo 10 princiapales cultivos
tabla_abs_limpia = tabla_abs_limpia.head(12)
# Generar totales por columna (tipo de riego)
tabla_abs_limpia.loc["Total"] = tabla_abs_limpia.sum()
# Mostrar tabla limpia
display(tabla_abs_limpia)



Riego,Aspersión,Cañon,Goteo,Manguera
Cultivo,,,,
Uchuva,0,0,0,0
Lechuga,9,0,0,0
Ajo,3,0,0,0
No Indica,2,0,0,0
Pasto Kikuyo,0,4,0,0
Pastos,2,0,0,0
Hortalizas Varias,3,0,0,0
Gulupa,1,0,0,0
Quinua,3,0,0,0


### Análisis del sistema de riego por cultivo

El análisis de la tabla de contingencia muestra que el sistema de riego más común en los registros analizados es el riego por aspersión, con una clara dominancia frente a otros métodos como goteo, cañón o manguera.

En el caso de la lechuga, un cultivo hortícola cuyo principal valor comercial es la hoja, resulta llamativo que el sistema de riego predominante sea la aspersión. Este método puede incrementar el riesgo de enfermedades bacterianas y fúngicas al mojar el follaje, además de aumentar la suciedad en las hojas. Desde un punto de vista técnico, sería esperable una mayor presencia del riego por goteo, dado que este mejora la eficiencia en el uso del agua, reduce el contacto del follaje con la humedad, permite fertiriego y una mejor sanidad del cultivo. Sin embargo, en los registros analizados no se observa ningún caso de riego por goteo para este cultivo.

Una situación similar se presenta en el ajo. Aunque el tamaño muestral es reducido, todos los registros corresponden a riego por aspersión. Esto contrasta con las recomendaciones técnicas, ya que el riego por goteo es generalmente preferido en este cultivo por dirigir el agua directamente a la base de la planta, mejorar la eficiencia hídrica, permitir la fertirrigación y evitar sobre riego, por tanto, reducir el riesgo de enfermedades asociadas al exceso de humedad y al encharcamiento del suelo, que puede provocar pudrición de los bulbos.

El pasto Kikuyo (Pennisetum clandestinum), utilizado principalmente como forraje en ganadería en climas fríos y zonas altas, presenta un patrón diferente. En este caso, el sistema de riego registrado es exclusivamente el riego por cañón, lo cual resulta coherente con la necesidad de aplicar riegos profundos y cubrir grandes áreas. En el caso de 'Otros pastos' y 'Maiz', aparecen asociados al riego por aspersión, un sistema viable cuando se realizan riegos más frecuentes y controlados (nota: no se especifica tipo de pastizal).

El tomate es el único cultivo que muestra variabilidad en los sistemas de riego, con registros de goteo y manguera. Esta diversidad sugiere prácticas de manejo menos estandarizadas o adaptadas a condiciones específicas del predio, como disponibilidad de infraestructura, escala de producción o topografía.

En general, el resto de los cultivos registrados presenta una fuerte predominancia del riego por aspersión. No obstante, en algunos casos —como hortalizas, quinua y cala— podría esperarse una mayor adopción del riego por goteo. Esto sugiere la necesidad de considerar la topografía y otras variables del terreno como posibles factores limitantes para la implementación de sistemas más eficientes y que mejoren las condiciones fitosanitarias para estos cultivos.

Finalmente, aunque la uchuva es el cultivo más frecuente en el dataset, lamentablemente no presenta información válida sobre el sistema de riego tras el proceso de limpieza de datos. Esta ausencia limita cualquier interpretación técnica sobre su manejo hídrico.

In [4]:
# Validando incidencia de la Topografia para los cultivos de 'Calas, Quinua, Hortalizas Varias y Gulupa'
# tabla de contingencia P(Topografia | Cultivo)

# Cultivos de interés
cultivos_interes = [
    "Calas",
    "Quinua",
    "Hortalizas Varias",
    "Gulupa"
]

# Tabla de contingencia P(Topografia | Cultivo) (frecuencias absolutas)
tabla_abs = (
    data
    .groupby("Cultivo")["Topografia"]
    .value_counts()
    .unstack(fill_value=0)
)

# Filtrar solo los cultivos de interés
tabla_abs_filtrada = tabla_abs.loc[
    tabla_abs.index.intersection(cultivos_interes)
]

display(tabla_abs_filtrada)


Topografia,Moderadamente ondulado,No indica,Ondulado,Pendiente,Pendiente fuerte,Pendiente moderada,Plano
Cultivo,,,,,,,
Calas,0,0,0,0,0,0,3
Gulupa,0,0,0,1,1,0,2
Hortalizas Varias,0,0,0,0,0,0,4
Quinua,0,0,0,0,0,0,3


In [5]:
# Validando incidencia del drenaje para los cultivos de 'Calas, Quinua, Hortalizas Varias y Gulupa'
# tabla de contingencia P(Drenaje | Cultivo)

# Cultivos de interés
cultivos_interes = [
    "Calas",
    "Quinua",
    "Hortalizas Varias",
    "Gulupa"
]

# Tabla de contingencia P(Topografia | Cultivo) (frecuencias absolutas)
tabla_abs = (
    data
    .groupby("Cultivo")["Drenaje"]
    .value_counts()
    .unstack(fill_value=0)
)

# Filtrar solo los cultivos de interés
tabla_abs_filtrada = tabla_abs.loc[
    tabla_abs.index.intersection(cultivos_interes)
]

display(tabla_abs_filtrada)

Drenaje,Buen drenaje,Mal drenaje,No indica,Regular drenaje
Cultivo,,,,
Calas,3,0,0,0
Gulupa,2,0,1,1
Hortalizas Varias,4,0,0,0
Quinua,3,0,0,0


Sin profundizar en las condiciones agronómicas óptimas de cada cultivo, los registros sugieren que, desde el punto de vista de la topografía y el drenaje, la implementación de sistemas de riego por goteo podría ser viable en varios casos. No obstante, la baja cantidad de registros disponibles limita notablemente esta inferencia.

In [6]:
# tabla de contigencia P(Drenaje | Cultivo) VER TODOS LOS CULTIVOS
# Generar tabla de contingencia no normalizada
tabla_abs = (  # (frecuencias absolutas)
    data 
    .groupby("Cultivo")["Drenaje"]
    .value_counts()
    .unstack(fill_value=0)
)
# Ordenar
orden_cultivos = tabla_abs.sum(axis=1).sort_values(ascending=False).index
tabla_abs = tabla_abs.loc[orden_cultivos]
# Mostrar tabla limpia
display(tabla_abs)

Drenaje,Buen drenaje,Mal drenaje,No indica,Regular drenaje
Cultivo,,,,
Uchuva,14,0,0,0
Lechuga,7,0,4,0
Ajo,4,0,3,0
No Indica,0,0,7,0
Pasto Kikuyo,7,0,0,0
Pastos,6,0,0,0
Hortalizas Varias,4,0,0,0
Gulupa,2,0,1,1
Quinua,3,0,0,0
